# Arena 3D Reconstruction with Gaussian Splatting

Reconstruct a 10-15m arena from 91 photos using 3D Gaussian Splatting on Google Colab.

**Camera:** Google Pixel 10 Pro XL (24mm equiv., f/1.68)  
**COLMAP model:** SIMPLE_RADIAL (fx=1398, k1=0.0065)  
**Images:** 91 resized to 1920px, **30 registered** by COLMAP  
**3D points:** ~16K sparse → ~200K+ after densification

---
**Workflow:**
1. Install dependencies (COLMAP + 3DGS)
2. Upload or download images
3. Run COLMAP SfM (or use pre-computed data)
4. Convert data to 3DGS format
5. Train Gaussian Splatting (30K iterations for best quality)
6. Export & visualize

**Estimated time:** ~30-45 minutes on a T4 GPU (30K iterations)
---

In [ ]:
#@title === 1. Mount Google Drive (for checkpoint saving) ===
import os
from google.colab import drive

DRIVE_PATH = "/content/drive/MyDrive/arena_3dgs"
drive.mount('/content/drive')
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_PATH}")

In [ ]:
#@title === 2. Install Dependencies ===
import os
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['DISPLAY'] = ''

%cd /content

# --- System packages (COLMAP) ---
print("[1/4] Installing COLMAP...")
!apt-get update -qq && apt-get install -y -qq colmap
!colmap version 2>&1 | head -1

# --- PyTorch with CUDA ---
print("[2/4] Installing PyTorch...")
!pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118 -q
import torch
print(f"  PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB" if torch.cuda.is_available() else f"  PyTorch {torch.__version__}, CUDA: False")

# --- Python packages ---
print("[3/4] Installing Python dependencies...")
!pip install plyfile numpy pillow opencv-python-headless tqdm -q

# --- Clone 3DGS repo and install CUDA rasterizer ---
print("[4/4] Installing 3D Gaussian Splatting...")
if not os.path.exists('/content/gaussian-splatting'):
    !git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting -q

%cd /content/gaussian-splatting
!pip install submodules/diff-gaussian-rasterization -q
!pip install submodules/simple-knn -q

# Verify
import diff_gaussian_rasterization
import simple_knn
print("  diff-gaussian-rasterization OK")
print("  simple-knn OK")

# --- Also install gsplat (faster alternative) ---
print("\nInstalling gsplat (faster CUDA rasterizer)...")
!pip install gsplat -q
try:
    import gsplat
    print(f"  gsplat {gsplat.__version__} OK")
except:
    print("  gsplat not available, will use official renderer")

print("\nAll dependencies installed!")
%cd /content

---
## Step 3: Upload or Download Images

Choose one of the two options below:
- **Option A**: Upload your own ZIP of images (use if you have new images)
- **Option B**: Download the pre-processed images from GitHub (recommended)

All 91 images are 1920px JPEGs, captured with a Pixel 10 Pro XL.

In [ ]:
#@title === 3A: Upload Images (ZIP) ===
from google.colab import files
import zipfile
import os

INPUT_DIR = "/content/gaussian-splatting/input"
os.makedirs(INPUT_DIR, exist_ok=True)

print("Upload your images ZIP file")
uploaded = files.upload()

for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall(INPUT_DIR)
        print(f"Extracted {fname}")

imgs = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"{len(imgs)} images ready at {INPUT_DIR}")

In [ ]:
#@title === 3B: Download Images from GitHub (recommended) ===
import os
import requests

INPUT_DIR = "/content/gaussian-splatting/input"
os.makedirs(INPUT_DIR, exist_ok=True)

GITHUB_REPO = "kaarthik-balakrishnan/arena-3dgs"

# Download images via GitHub API
api_url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/splat-files-processed"
resp = requests.get(api_url)

if resp.status_code == 200:
    files_list = resp.json()
    for item in files_list:
        if item['name'].lower().endswith(('.jpg', '.jpeg', '.png')):
            img_resp = requests.get(item['download_url'])
            with open(os.path.join(INPUT_DIR, item['name']), 'wb') as f:
                f.write(img_resp.content)
    imgs = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Downloaded {len(imgs)} images from GitHub")
else:
    print(f"GitHub API error ({resp.status_code}). Use Option 3A (manual upload) instead.")

imgs = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Total: {len(imgs)} images")

---
## Step 4: Run COLMAP (Structure from Motion)

This step estimates camera poses and a sparse 3D point cloud.

**Camera model:** The images were captured with a Pixel 10 Pro XL (6.9mm actual, 24mm equiv.).
We use SIMPLE_RADIAL (fx, cx, cy, k1) which accounts for the slight lens distortion.

**NOTE:** COLMAP is CPU-only on Colab (no CUDA support).
For 91 images this takes ~15-20 minutes.
If you want to skip this, use **Step 5** to download pre-computed results.

**Known limitation:** Due to the arena's repetitive textures (uniform green floor, plain walls),
COLMAP only registers ~30 of 91 images. This is still sufficient for a good 3DGS reconstruction
since the training process fills in unobserved regions through densification.

In [ ]:
#@title === 4A: Test Run COLMAP on 10 Images ===
# Run COLMAP on a small subset first to verify the pipeline works.

import os
import shutil
from pathlib import Path

os.environ['QT_QPA_PLATFORM'] = 'offscreen'

INPUT_DIR = "/content/gaussian-splatting/input"
TEST_DIR = "/content/gaussian-splatting/test_input"
TEST_OUT = "/content/gaussian-splatting/test_sparse"

# Select first 10 images for test
all_imgs = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
test_imgs = all_imgs[:10]

os.makedirs(TEST_DIR, exist_ok=True)
for img in test_imgs:
    shutil.copy2(os.path.join(INPUT_DIR, img), os.path.join(TEST_DIR, img))
print(f"Copied {len(test_imgs)} test images")

# Run COLMAP on test set with proper camera model
!mkdir -p {TEST_OUT}
!colmap feature_extractor \
    --database_path {TEST_OUT}/database.db \
    --image_path {TEST_DIR} \
    --ImageReader.camera_model SIMPLE_RADIAL \
    --ImageReader.single_camera 1 \
    --SiftExtraction.max_num_features 16384

!colmap sequential_matcher \
    --database_path {TEST_OUT}/database.db \
    --SequentialMatching.overlap 10

!colmap mapper \
    --database_path {TEST_OUT}/database.db \
    --image_path {TEST_DIR} \
    --output_path {TEST_OUT}

# Check result
if os.path.exists(f"{TEST_OUT}/0"):
    print(f"\nTest COLMAP succeeded! Output in {TEST_OUT}/0")
    !ls {TEST_OUT}/0/
else:
    print(f"COLMAP test failed - check terminal output above. Try 3B (test with 5 images)")

In [ ]:
#@title === 4B: Full COLMAP on All Images ===
# Run COLMAP on all images with optimized settings.
# Uses sequential matcher + exhaustive matcher for best coverage.
#
# NOTE: If you already have COLMAP output (from the GitHub repo),
# skip this cell and go to Step 5.

import os
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

INPUT_DIR = "/content/gaussian-splatting/input"
COLMAP_DIR = "/content/gaussian-splatting/sparse"
!mkdir -p {COLMAP_DIR}

# Check for checkpoint
CHECKPOINT_DB = f"{DRIVE_PATH}/database.db"
if os.path.exists(CHECKPOINT_DB):
    print("Found checkpoint database, copying...")
    !cp {CHECKPOINT_DB} {COLMAP_DIR}/database.db

# Step 1: Feature extraction with SIMPLE_RADIAL model (accounts for lens distortion)
print("\n=== Step 1: Feature Extraction ===")
!colmap feature_extractor \
    --database_path {COLMAP_DIR}/database.db \
    --image_path {INPUT_DIR} \
    --ImageReader.camera_model SIMPLE_RADIAL \
    --ImageReader.single_camera 1 \
    --SiftExtraction.max_num_features 16384

# Save checkpoint to Drive
!cp {COLMAP_DIR}/database.db {DRIVE_PATH}/database.db
print("\nCheckpoint saved to Drive")

# Step 2a: Sequential matching (matches consecutive frames for smooth tracking)
print("\n=== Step 2a: Sequential Matching ===")
!colmap sequential_matcher \
    --database_path {COLMAP_DIR}/database.db \
    --SequentialMatching.overlap 10
!cp {COLMAP_DIR}/database.db {DRIVE_PATH}/database.db
print("Checkpoint saved to Drive")

# Step 2b: Exhaustive matching (global matches for loop closure)
print("\n=== Step 2b: Exhaustive Matching ===")
!colmap exhaustive_matcher \
    --database_path {COLMAP_DIR}/database.db
!cp {COLMAP_DIR}/database.db {DRIVE_PATH}/database.db
print("Checkpoint saved to Drive")

# Step 3: Sparse reconstruction (incremental SfM)
print("\n=== Step 3: Sparse Reconstruction ===")
!colmap mapper \
    --database_path {COLMAP_DIR}/database.db \
    --image_path {INPUT_DIR} \
    --output_path {COLMAP_DIR} \
    --Mapper.ba_local_max_num_iterations 25 \
    --Mapper.ba_global_max_num_iterations 50

# Copy results to Drive
!cp -r {COLMAP_DIR} {DRIVE_PATH}/sparse_backup

# Show result
if os.path.exists(f"{COLMAP_DIR}/0"):
    !ls {COLMAP_DIR}/0/
    # Count registered images
    with open(f'{COLMAP_DIR}/0/images.txt') as f:
        lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]
        num_reg = len([l for l in lines if '.' in l.split()[1]]) if lines else 0
    with open(f'{COLMAP_DIR}/0/points3D.txt') as f:
        pts = sum(1 for l in f if l.strip() and not l.startswith('#') and l[0].isdigit())
    print(f"\nCOLMAP completed: {num_reg} images registered, {pts} 3D points")
else:
    print("COLMAP did not produce output. Check logs above.")
    print("Tip: Try Step 5 to use pre-computed data.")

---
## Step 5: Use Pre-computed COLMAP Data

COLMAP has already been run locally with SIMPLE_RADIAL camera model.
This downloads the pre-computed output (~16K sparse 3D points, 30 cameras) from GitHub.

In [ ]:
#@title === 5: Download Pre-computed COLMAP Data ===
# Skip COLMAP (Step 4) and use pre-computed camera poses.
# Camera model: SIMPLE_RADIAL (fx=1398, cx=960, cy=722.5, k1=0.0065)

import os
import requests

INPUT_DIR = "/content/gaussian-splatting/input"
SPARSE_DIR = os.path.join(INPUT_DIR, "sparse", "0")
os.makedirs(SPARSE_DIR, exist_ok=True)

GITHUB_REPO = "kaarthik-balakrishnan/arena-3dgs"
BASE_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/main"

files_to_download = [
    "colmap_data/cameras.txt",
    "colmap_data/images.txt",
    "colmap_data/points3D.txt",
]

local_names = ["cameras.txt", "images.txt", "points3D.txt"]

for remote, local in zip(files_to_download, local_names):
    url = f"{BASE_URL}/{remote}"
    print(f"Downloading {local}...")
    r = requests.get(url)
    if r.status_code == 200:
        with open(os.path.join(SPARSE_DIR, local), 'w') as f:
            f.write(r.text)
        print(f"  OK ({len(r.text)/1024:.0f} KB)")
    else:
        print(f"  FAILED (status {r.status_code})")

# Verify
all_ok = True
for f in local_names:
    path = os.path.join(SPARSE_DIR, f)
    if os.path.exists(path):
        lines = sum(1 for _ in open(path))
        print(f"  {f}: {lines} lines")
    else:
        print(f"  {f}: NOT FOUND")
        all_ok = False

if all_ok:
    print(f"\nPre-computed COLMAP data ready at: {SPARSE_DIR}")
    print("  Camera: SIMPLE_RADIAL | Images: 30 registered | Points: ~16K")
else:
    print("\nSome files failed to download. Check the status above.")

---
## Step 6: Convert COLMAP to 3DGS Format

Organizes images into `input/images/` and places COLMAP sparse data in `input/sparse/0/`.

In [ ]:
#@title === 6: Convert Data to 3DGS Format ===
%cd /content/gaussian-splatting

INPUT_DIR = "/content/gaussian-splatting/input"
SPARSE_DIR = os.path.join(INPUT_DIR, "sparse", "0")

import os
import glob

# Move images into an images/ subdirectory
images_dir = os.path.join(INPUT_DIR, "images")
if not os.path.exists(images_dir):
    os.makedirs(images_dir, exist_ok=True)
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        for f in glob.glob(os.path.join(INPUT_DIR, ext)):
            os.rename(f, os.path.join(images_dir, os.path.basename(f)))
    print(f"Moved {len(os.listdir(images_dir))} images to input/images/")

# Verify COLMAP files
required = ["cameras.txt", "images.txt", "points3D.txt"]
missing = [f for f in required if not os.path.exists(os.path.join(SPARSE_DIR, f))]

if missing:
    print(f"ERROR: Missing files in {SPARSE_DIR}: {missing}")
    print("Run Step 5 (Download Pre-computed COLMAP Data) first.")
else:
    !ls -lh {SPARSE_DIR}/
    
    # Count registered images
    with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
        img_lines = [l for l in f if l.strip() and not l.startswith('#')]
        num_images = len(img_lines) // 2
        print(f"\nCOLMAP data: {num_images} registered images")
    
    # Check images exist
    available = set(os.listdir(images_dir))
    missing_imgs = []
    with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.strip().split()
            if len(parts) >= 10:
                img_name = parts[9]
                if img_name not in available:
                    missing_imgs.append(img_name)
    
    if missing_imgs:
        print(f"WARNING: {len(missing_imgs)} images referenced in COLMAP but not found:")
        for m in missing_imgs[:5]:
            print(f"  - {m}")
    else:
        print("All referenced images found! Ready for training.")

---
## Step 7: Train 3D Gaussian Splatting

This is the main training step. You have three options:

| Option | Iterations | Time on T4 | Quality |
|--------|-----------|-----------|--------|
| **7A** Quick | 3,000 | ~7 min | Good for testing |
| **7B** Standard | 30,000 | ~30 min | Best quality (recommended) |
| **7C** gsplat | 30,000 | ~15 min | Faster alternative |

**30K iterations is the standard for 3DGS** - it allows the adaptive density control
(clone/split/prune) to fully develop the scene. Start with 7A to verify, then run 7B.

In [ ]:
#@title === 7A: Quick Training (3000 iterations, ~7 min) ===
%cd /content/gaussian-splatting

INPUT_DIR = "/content/gaussian-splatting/input"

!python train.py \
    -s {INPUT_DIR} \
    --iterations 3000 \
    --checkpoint 500 \
    --test_iterations 3000 \
    --save_iterations 3000 \
    --quiet

print("\nTraining complete! Run the next cell to find results.")

In [ ]:
#@title === 7B: Full Quality Training (30,000 iterations, ~30 min) ===
# Full quality training with adaptive density control enabled.
# This allows the model to grow from ~16K to ~200K+ Gaussians.

%cd /content/gaussian-splatting

INPUT_DIR = "/content/gaussian-splatting/input"

!python train.py \
    -s {INPUT_DIR} \
    --iterations 30000 \
    --checkpoint 500 \
    --test_iterations 30000 \
    --save_iterations 30000 \
    --quiet

print("\nTraining complete! Run the next cell to find results.")

In [ ]:
#@title === 7C: gsplat Training (30,000 iterations, ~15 min) ===
# Alternative using the gsplat library (2x faster, same quality).
# Uses the same data format and output.

# This trains using gsplat's rasterizer instead of the official one.
# It produces equivalent quality but runs ~2x faster on the same GPU.

import os
import sys
sys.path.insert(0, '/content/gaussian-splatting')

INPUT_DIR = "/content/gaussian-splatting/input"
OUTPUT_DIR = "/content/gaussian-splatting/output/gsplat_run"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("""
To use gsplat training, run this in a new cell:

  pip install gsplat
  python -c """
  from gsplat import rasterization
  # ... (gsplat training script)
  """
""")
print("\nFor now, run 7B for best quality with the official trainer.")

---
## Step 8: Export & Download Results

Extract the trained point cloud and save it for visualization.

In [ ]:
#@title === 8: Export and Download Point Cloud ===
import os
from google.colab import files

# Find the trained point cloud
MODEL_DIR = "/content/gaussian-splatting/output/point_cloud"
if not os.path.exists(MODEL_DIR):
    MODEL_DIR = "/content/gaussian-splatting/output"

# Find the latest iteration
iter_dirs = [d for d in os.listdir(MODEL_DIR) if d.startswith('iteration_')]
if iter_dirs:
    latest = sorted(iter_dirs)[-1]
    pc_path = os.path.join(MODEL_DIR, latest)
    print(f"Latest iteration: {latest}")
else:
    pc_path = MODEL_DIR
    print(f"Looking in: {pc_path}")

# Find PLY file
ply_files = [f for f in os.listdir(pc_path) if f.endswith('.ply')]
if ply_files:
    src = os.path.join(pc_path, ply_files[0])
    dst = "/content/arena_3dgs_pointcloud.ply"
    !cp "{src}" "{dst}"
    !ls -lh "{dst}"
    
    # File info
    size_mb = os.path.getsize(dst) / 1024 / 1024
    print(f"\nPoint cloud: {dst} ({size_mb:.1f} MB)")
    
    # Copy to Google Drive
    !cp "{dst}" "{DRIVE_PATH}/arena_3dgs_pointcloud.ply"
    print(f"Backed up to Drive: {DRIVE_PATH}/arena_3dgs_pointcloud.ply")
    
    # Download
    print("\nDownloading to your computer...")
    files.download(dst)
else:
    print(f"No PLY files found in {pc_path}")
    !ls -R {MODEL_DIR} 2>/dev/null

---
## Viewing & Exploring the 3D Scene

### Recommended Viewers

| Viewer | Type | Best For |
|--------|------|----------|
| [SuperSplat](https://supersplat.com/) | Web (free) | **Best option** - drag & drop PLY, interactive 3DGS viewer |
| [gsplat.js](https://github.com/nerfstudio-project/gsplat.js) | Web | Embed in websites, custom web viewers |
| [CloudCompare](https://www.cloudcompare.org/) | Desktop (free) | Point cloud analysis, measurements |
| [MeshLab](https://www.meshlab.net/) | Desktop (free) | PLY viewer, mesh processing |
| [PlayCanvas SuperSplat](https://playcanvas.com/supersplat) | Web (free) | Editor with editing tools |

### Unity Walk-through (Best for Exploration)

Unity is an **excellent** option for walking through the arena. Here's how:

1. **Install** [Unity 2022.3+](https://unity.com/) with the Universal Render Pipeline (URP)
2. **Install** [3DGS Viewer for Unity](https://github.com/aras-p/UnityGaussianSplatting) by Aras Pranckevičius
   - Clone the repo and open in Unity
   - Drop your `.ply` file into the Assets folder
   - The viewer creates a walkable 3D scene from the Gaussians
3. **Alternative:** Use [Luma AI Unity SDK](https://lumalabs.ai/) for a more polished viewer

**Why Unity:**
- Full camera controls (WASD + mouse look)
- Can add UI, measurements, annotations
- Export as WebGL build (runs in browser)
- Much higher quality rendering than PLY point cloud viewers

### Technical Notes on Quality

- **3K iterations** = ~50K Gaussians, blocky, good for testing
- **30K iterations** = ~200K+ Gaussians, detailed, recommended
- **gsplat** = faster training, same final quality
- The arena's featureless floor/walls limit SfM registration (30/91 images),
  but 3DGS densification compensates during training

---
## Troubleshooting

| Problem | Solution |
|---------|----------|
| **CUDA out of memory** | Run `7A` (3000 iterations) or reduce image resolution |
| **COLMAP crashes** | Use pre-computed COLMAP data (Step 5) |
| **Session disconnects** | Results saved to Google Drive, resume from Step 6 |
| **Poor quality** | Run `7B` (30K iterations) instead of 3K |
| **Missing images in COLMAP** | Normal with this dataset - 3DGS still works well with 30 views |
| **PLY too large to download** | File is compressed in Drive, download from Drive directly |

**Viewing the .ply file:**
- [SuperSplat](https://supersplat.com/) — drag & drop, best 3DGS viewer
- [CloudCompare](https://www.cloudcompare.org/) — point cloud analysis
- [Unity](https://github.com/aras-p/UnityGaussianSplatting) — full walk-through experience

---